<a href="https://colab.research.google.com/github/10dimensions/large-notebook-repository/blob/master/tinyllama_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# 🛠️ STEP 1: INSTALL DEPENDENCIES
# ==========================================
# Run this once at the start of the notebook
!pip install -q transformers accelerate bitsandbytes datasets scipy tabulate --upgrade

In [5]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.9/188.9 MB 7.0 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0


In [1]:
# ==========================================
# 🚀 STEP 2: IMPORT & CONFIGURATION
# ==========================================
import torch
import gc
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tabulate import tabulate
import warnings
warnings.filterwarnings("ignore")

# Check GPU Availability (Critical for Colab)
if not torch.cuda.is_available():
    raise RuntimeError("⚠️  No GPU detected! Please go to Runtime > Change Runtime Type > Select GPU.")

DEVICE = "cuda"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SAMPLE_PROMPT = "The future of artificial intelligence is"
MAX_NEW_TOKENS = 30  # Reduced for faster Colab demo

print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
print(f"✅ CUDA Version: {torch.version.cuda}")

✅ GPU Detected: Tesla T4
✅ CUDA Version: 12.8


In [2]:
# ==========================================
# 🛠️ STEP 3: BENCHMARKING UTILS
# ==========================================
def get_vram_usage():
    """Returns VRAM usage in MB"""
    return torch.cuda.memory_allocated() / 1024**2

def calculate_perplexity(model, tokenizer, text):
    """Fast perplexity check on a small sample"""
    model.eval()
    encodings = tokenizer(text[:512], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(encodings.input_ids, labels=encodings.input_ids)
        loss = outputs.loss
    return torch.exp(loss).item()

def measure_speed(model, tokenizer, prompt):
    """Measures Tokens Per Second (TPS)"""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            inputs.input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    end = time.time()

    generated_tokens = output.shape[1] - inputs.input_ids.shape[1]
    tps = generated_tokens / (end - start)
    return tps

def cleanup():
    """Aggressive memory cleanup for Colab"""
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)

In [9]:
# ==========================================
# 🛠️ STEP 4: COMPRESSION FUNCTIONS
# ==========================================

def load_fp16_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return model, tokenizer

def load_4bit_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    return model, tokenizer

def apply_pruning(model, amount=0.2):
    """Simple magnitude pruning on linear layers"""
    print(f"  ✂️  Pruning {amount*100}% of weights...")
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            weight = module.weight.data
            threshold = torch.quantile(torch.abs(weight).float(), amount)
            mask = torch.abs(weight) > threshold
            module.weight.data = weight * mask.float()
    return model

In [10]:
# ==========================================
# 🏃 STEP 5: RUN PIPELINE
# ==========================================
results = []

print("\n" + "="*50)
print("🚀 STARTING LLM COMPRESSION BENCHMARK")
print("="*50)

# --- 1. Baseline (FP16) ---
print("\n[1/3] Loading Baseline (FP16)...")
try:
    model_fp16, tokenizer = load_fp16_model()
    vram = get_vram_usage()
    tps = measure_speed(model_fp16, tokenizer, SAMPLE_PROMPT)
    # Perplexity takes time, using a short sample for Colab speed
    ppl = calculate_perplexity(model_fp16, tokenizer, SAMPLE_PROMPT)

    results.append(["TinyLlama (FP16)", f"{vram:.1f} MB", f"{tps:.2f}", f"{ppl:.2f}"])
    print(f"  ✅ Done | VRAM: {vram:.1f}MB | Speed: {tps:.2f} TPS")

    del model_fp16
    cleanup()
except Exception as e:
    print(f"  ❌ Failed: {e}")
    results.append(["TinyLlama (FP16)", "Error", "Error", "Error"])

# --- 2. Quantized (4-bit) ---
print("\n[2/3] Loading Quantized (4-bit NF4)...")
try:
    model_4bit, tokenizer = load_4bit_model()
    vram = get_vram_usage()
    tps = measure_speed(model_4bit, tokenizer, SAMPLE_PROMPT)
    ppl = calculate_perplexity(model_4bit, tokenizer, SAMPLE_PROMPT)

    results.append(["TinyLlama (4-bit)", f"{vram:.1f} MB", f"{tps:.2f}", f"{ppl:.2f}"])
    print(f"  ✅ Done | VRAM: {vram:.1f}MB | Speed: {tps:.2f} TPS")

    del model_4bit
    cleanup()
except Exception as e:
    print(f"  ❌ Failed: {e}")
    results.append(["TinyLlama (4-bit)", "Error", "Error", "Error"])

# --- 3. Pruned (FP16 + Sparse) ---
print("\n[3/3] Loading Pruned Model (FP16 + 20% Sparse)...")
try:
    model_pruned, tokenizer = load_fp16_model()
    model_pruned = apply_pruning(model_pruned, amount=0.2)

    vram = get_vram_usage()
    tps = measure_speed(model_pruned, tokenizer, SAMPLE_PROMPT)
    ppl = calculate_perplexity(model_pruned, tokenizer, SAMPLE_PROMPT)

    results.append(["TinyLlama (Pruned)", f"{vram:.1f} MB", f"{tps:.2f}", f"{ppl:.2f}"])
    print(f"  ✅ Done | VRAM: {vram:.1f}MB | Speed: {tps:.2f} TPS")

    del model_pruned
    cleanup()
except Exception as e:
    print(f"  ❌ Failed: {e}")
    results.append(["TinyLlama (Pruned)", "Error", "Error", "Error"])


🚀 STARTING LLM COMPRESSION BENCHMARK

[1/3] Loading Baseline (FP16)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✅ Done | VRAM: 4204.5MB | Speed: 31.73 TPS

[2/3] Loading Quantized (4-bit NF4)...
  ❌ Failed: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

[3/3] Loading Pruned Model (FP16 + 20% Sparse)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✂️  Pruning 20.0% of weights...
  ❌ Failed: quantile() input tensor is too large


In [11]:
# ==========================================
# 📊 STEP 6: DISPLAY RESULTS
# ==========================================
print("\n" + "="*50)
print("📊 FINAL BENCHMARK REPORT")
print("="*50)
print(tabulate(results, headers=["Model", "VRAM Usage", "Speed (TPS)", "Perplexity"], tablefmt="grid"))

print("\n💡 Colab Tips:")
print("1. VRAM Usage shows memory efficiency (4-bit saves ~70% memory).")
print("2. Speed (TPS) depends on memory bandwidth (4-bit is usually faster).")
print("3. Perplexity measures accuracy (Lower is better).")
print("4. If you get OOM errors, try restarting the runtime (Runtime > Restart).")


📊 FINAL BENCHMARK REPORT
+--------------------+--------------+---------------+--------------+
| Model              | VRAM Usage   | Speed (TPS)   | Perplexity   |
+====================+==============+===============+==============+
| TinyLlama (FP16)   | 4204.5 MB    | 31.73         | 80.99        |
+--------------------+--------------+---------------+--------------+
| TinyLlama (4-bit)  | Error        | Error         | Error        |
+--------------------+--------------+---------------+--------------+
| TinyLlama (Pruned) | Error        | Error         | Error        |
+--------------------+--------------+---------------+--------------+

💡 Colab Tips:
1. VRAM Usage shows memory efficiency (4-bit saves ~70% memory).
2. Speed (TPS) depends on memory bandwidth (4-bit is usually faster).
3. Perplexity measures accuracy (Lower is better).
4. If you get OOM errors, try restarting the runtime (Runtime > Restart).
